In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/amazon-reviews/amazon_review_polarity_csv.tgz
/kaggle/input/amazon-reviews/train.csv
/kaggle/input/amazon-reviews/test.csv


In [2]:
import numpy as np
import pandas as pd
import random
import string
import warnings
warnings.filterwarnings("ignore")

In [3]:
!pip install gensim fasttext scikit-learn

In [4]:
import gensim.downloader as api
from gensim.models import Word2Vec, FastText as GensimFastText
from sklearn.metrics.pairwise import cosine_similarity
from gensim.utils import simple_preprocess


In [6]:
df = pd.read_csv("/kaggle/input/amazon-reviews/train.csv", header=None, names=["polarity", "title", "text"])

In [7]:
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

In [8]:
df["review"] = df["title"].fillna("") + " " + df["text"].fillna("")

In [9]:
reviews = df["review"].tolist()

In [10]:
print(f"Loaded {len(reviews)} reviews")

Loaded 5000 reviews


In [11]:
tokenized_reviews = [simple_preprocess(r) for r in reviews]

In [12]:
print("\n" + "="*60)
print("Q1: FastText Embeddings & Semantic Similarity Retrieval")
print("="*60)



Q1: FastText Embeddings & Semantic Similarity Retrieval


In [13]:
# Train FastText on the corpus (pretrained alternative: api.load("fasttext-wiki-news-subwords-300"))
print("Training FastText model...")
ft_model = GensimFastText(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10
)
print("FastText model trained.")


Training FastText model...
FastText model trained.


In [14]:
def get_review_embedding_ft(tokens, model):
    """Mean-pool word vectors for all tokens."""
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)


In [15]:
print("Computing review embeddings...")
review_embeddings = np.array([
    get_review_embedding_ft(tok, ft_model) for tok in tokenized_reviews
])


Computing review embeddings...


In [16]:
def retrieve_top_k(query: str, model, review_embs, reviews_list, k=5):
    """Embed query and return top-k most similar reviews."""
    query_tokens = simple_preprocess(query)
    query_emb = get_review_embedding_ft(query_tokens, model).reshape(1, -1)
    sims = cosine_similarity(query_emb, review_embs)[0]
    top_k_idx = sims.argsort()[::-1][:k]
    return [(reviews_list[i], round(float(sims[i]), 4)) for i in top_k_idx]


In [17]:
query = "battery life is very poor"
print(f"\nQuery: \"{query}\"")
print("\nTop-5 Semantically Similar Reviews:")
results = retrieve_top_k(query, ft_model, review_embeddings, reviews)
for rank, (rev, score) in enumerate(results, 1):
    print(f"\n[{rank}] Similarity: {score}")
    print(f"    {rev[:200]}...")


Query: "battery life is very poor"

Top-5 Semantically Similar Reviews:

[1] Similarity: 0.8792
    As advertised This under-rug padding was delivered in good condition and is of good quality. I recommend this product....

[2] Similarity: 0.8598
    An album of poor quality This is plainly an album of poor quality. Not only has Russell Watson a course and untrained voice, his performances are non-stylish, unsubtle and unidiomatic. The orchestral ...

[3] Similarity: 0.8568
    this is a gem the video is pretty tough but this is a gemrealise it is 1955 live tvit is a rare live performance of the great noel cowardand a great pairing with mary martinthis is a real charm to wat...

[4] Similarity: 0.8547
    For Mucha lovers This is a great biography of Mucha. Condensed and easy to read. Contains beautiful images and is worth the price....

[5] Similarity: 0.8412
    Exciting plot, great characterization ... a winner! Kinsey does it again! She's the queen of the supersleuths ... "B is for

In [18]:
print("\n" + "="*60)
print("Q2: Noisy Reviews — Word2Vec vs FastText Comparison")
print("="*60)



Q2: Noisy Reviews — Word2Vec vs FastText Comparison


In [19]:
def introduce_noise(text, noise_prob=0.3):
    """Randomly introduce spelling mistakes and noisy words."""
    words = text.split()
    noisy_words = []
    for word in words:
        r = random.random()
        if r < noise_prob / 3:
            # Random character substitution
            if len(word) > 2:
                idx = random.randint(1, len(word) - 2)
                word = word[:idx] + random.choice(string.ascii_lowercase) + word[idx+1:]
        elif r < 2 * noise_prob / 3:
            # Duplicate a character
            if len(word) > 1:
                idx = random.randint(0, len(word) - 1)
                word = word[:idx] + word[idx] + word[idx:]
        elif r < noise_prob:
            # Insert random noise word
            noisy_words.append(random.choice(["xzq", "zzz", "asdf", "qwerty"]))
        noisy_words.append(word)
    return " ".join(noisy_words)

In [20]:
random.seed(42)
sample_reviews = reviews[:500]
noisy_reviews = [introduce_noise(r) for r in sample_reviews]

In [21]:
print("\nOriginal review (first 150 chars):")
print(sample_reviews[0][:150])
print("\nNoisy version (first 150 chars):")
print(noisy_reviews[0][:150])


Original review (first 150 chars):
Expensive Junk This product consists of a piece of thin flexible insulating material, adhesive backed velcro and white electrical tape.Problems:1. Ins

Noisy version (first 150 chars):
Expensive Juhk xzq This product consists of a phece of twin flexible insulating asdf material, adhesive bawked velcro zzz and white electrical tale.Pr


In [22]:
tokenized_noisy = [simple_preprocess(r) for r in noisy_reviews]
tokenized_original = [simple_preprocess(r) for r in sample_reviews]


In [23]:
print("\nTraining Word2Vec on clean corpus...")
w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10
)



Training Word2Vec on clean corpus...


In [24]:
def oov_rate(tokenized_texts, model, model_type="w2v"):
    """Compute Out-Of-Vocabulary rate for a list of tokenized texts."""
    total, oov = 0, 0
    for tokens in tokenized_texts:
        for tok in tokens:
            total += 1
            if model_type == "w2v":
                if tok not in model.wv:
                    oov += 1
            else:  # FastText handles OOV via subwords
                try:
                    _ = model.wv[tok]
                except KeyError:
                    oov += 1
    return oov / total if total > 0 else 0

In [25]:
oov_w2v_clean  = oov_rate(tokenized_original, w2v_model, "w2v")
oov_ft_clean   = oov_rate(tokenized_original, ft_model,  "ft")
oov_w2v_noisy  = oov_rate(tokenized_noisy,    w2v_model, "w2v")
oov_ft_noisy   = oov_rate(tokenized_noisy,    ft_model,  "ft")

In [26]:
print("\n── OOV Rate Comparison ──")
print(f"{'Model':<12} {'Clean Reviews':>15} {'Noisy Reviews':>15}")
print("-" * 44)
print(f"{'Word2Vec':<12} {oov_w2v_clean:>14.2%} {oov_w2v_noisy:>14.2%}")
print(f"{'FastText':<12} {oov_ft_clean:>14.2%} {oov_ft_noisy:>14.2%}")


── OOV Rate Comparison ──
Model          Clean Reviews   Noisy Reviews
--------------------------------------------
Word2Vec              0.00%         24.08%
FastText              0.00%          0.00%


In [27]:
noisy_query = "batttery lif is verry pooor"
clean_query  = "battery life is very poor"
print(f"\nClean query : \"{clean_query}\"")
print(f"Noisy query : \"{noisy_query}\"")


Clean query : "battery life is very poor"
Noisy query : "batttery lif is verry pooor"


In [28]:
def embed_query(query, model):
    tokens = simple_preprocess(query)
    vecs = []
    for t in tokens:
        try:
            vecs.append(model.wv[t])
        except KeyError:
            pass
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

In [29]:
clean_emb_w2v = embed_query(clean_query,  w2v_model)
noisy_emb_w2v = embed_query(noisy_query,  w2v_model)
clean_emb_ft  = embed_query(clean_query,  ft_model)
noisy_emb_ft  = embed_query(noisy_query,  ft_model)

In [30]:
sim_w2v = cosine_similarity([clean_emb_w2v], [noisy_emb_w2v])[0][0]
sim_ft  = cosine_similarity([clean_emb_ft],  [noisy_emb_ft])[0][0]

In [31]:
print(f"\nCosine similarity (clean vs noisy query):")
print(f"  Word2Vec : {sim_w2v:.4f}  ← lower = hurt by noise")
print(f"  FastText : {sim_ft:.4f}  ← higher = robust via subwords")    


Cosine similarity (clean vs noisy query):
  Word2Vec : 0.7378  ← lower = hurt by noise
  FastText : 0.9555  ← higher = robust via subwords


In [32]:
print("\n" + "="*60)
print("Q3: Word Coverage — Word2Vec vs FastText")
print("="*60)


Q3: Word Coverage — Word2Vec vs FastText


In [33]:
all_tokens = [tok for tokens in tokenized_reviews for tok in tokens]
unique_words = set(all_tokens)
print(f"Total unique words in dataset: {len(unique_words)}")

Total unique words in dataset: 22962


In [34]:
w2v_vocab = set(w2v_model.wv.key_to_index.keys())
ft_vocab  = set(ft_model.wv.key_to_index.keys())

In [35]:
w2v_covered = unique_words & w2v_vocab
ft_covered  = unique_words & ft_vocab

In [36]:
w2v_oov = unique_words - w2v_vocab
ft_oov  = unique_words - ft_vocab 

In [37]:
print(f"\n── Vocabulary Coverage ──")
print(f"{'Metric':<35} {'Word2Vec':>10} {'FastText':>10}")
print("-" * 57)
print(f"{'Vocab size (model)':<35} {len(w2v_vocab):>10,} {len(ft_vocab):>10,}")
print(f"{'Dataset words covered (exact)':<35} {len(w2v_covered):>10,} {len(ft_covered):>10,}")
print(f"{'Dataset words NOT in vocab':<35} {len(w2v_oov):>10,} {len(ft_oov):>10,}")
print(f"{'Coverage %':<35} {len(w2v_covered)/len(unique_words):>9.1%} {len(ft_covered)/len(unique_words):>9.1%}")



── Vocabulary Coverage ──
Metric                                Word2Vec   FastText
---------------------------------------------------------
Vocab size (model)                      22,962     22,962
Dataset words covered (exact)           22,962     22,962
Dataset words NOT in vocab                   0          0
Coverage %                             100.0%    100.0%


In [38]:
print("\n── FastText OOV Embedding Demonstration ──")
example_oov_words = list(w2v_oov)[:5]  # words Word2Vec doesn't know
print(f"Sample words unknown to Word2Vec: {example_oov_words}")
print("\nFastText embedding behavior for these words:")
for word in example_oov_words:
    try:
        vec = ft_model.wv[word]
        in_ft_vocab = word in ft_vocab
        print(f"  '{word}': FastText returns vector (in_vocab={in_ft_vocab}), "
              f"norm={np.linalg.norm(vec):.4f}")
    except KeyError:
        print(f"  '{word}': FastText also fails (very unusual)")



── FastText OOV Embedding Demonstration ──
Sample words unknown to Word2Vec: []

FastText embedding behavior for these words:


In [39]:
# Show truly OOV words (misspellings not in training data)
test_oov = ["battttery", "amazingg", "terribl3", "wrst", "coool"]
print(f"\nHandling of artificially OOV words:")
print(f"{'Word':<15} {'W2V':<25} {'FastText'}")
print("-" * 60)
for word in test_oov:
    w2v_has = "IN vocab" if word in w2v_model.wv else "NOT in vocab (zero/skip)"
    try:
        vec = ft_model.wv[word]
        ft_has = f"Returns vector (norm={np.linalg.norm(vec):.3f})"
    except KeyError:
        ft_has = "NOT in vocab"
    print(f"{word:<15} {w2v_has:<25} {ft_has}")



Handling of artificially OOV words:
Word            W2V                       FastText
------------------------------------------------------------
battttery       NOT in vocab (zero/skip)  Returns vector (norm=3.093)
amazingg        NOT in vocab (zero/skip)  Returns vector (norm=3.160)
terribl3        NOT in vocab (zero/skip)  Returns vector (norm=2.780)
wrst            NOT in vocab (zero/skip)  Returns vector (norm=5.811)
coool           NOT in vocab (zero/skip)  Returns vector (norm=3.515)


In [40]:
print("\n── Summary ──")
print("""
Word2Vec:
  - Fixed vocabulary; cannot embed words not seen during training.
  - High OOV rate on noisy/misspelled text.
  - Similarity between clean and noisy queries is lower.

FastText:
  - Uses character n-grams; generates vectors for any word (even OOV).
  - Robust to spelling mistakes and morphological variants.
  - Similarity between clean and noisy queries is higher.
  - Better suited for noisy real-world text (reviews, social media, etc.).
""")


── Summary ──

Word2Vec:
  - Fixed vocabulary; cannot embed words not seen during training.
  - High OOV rate on noisy/misspelled text.
  - Similarity between clean and noisy queries is lower.

FastText:
  - Uses character n-grams; generates vectors for any word (even OOV).
  - Robust to spelling mistakes and morphological variants.
  - Similarity between clean and noisy queries is higher.
  - Better suited for noisy real-world text (reviews, social media, etc.).

